### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"]= os.getenv("OPENAI_API_KEY")
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

#### How to use tools?


1. Usage with init_chat_model()

Here first we can initiate the model and add the tool later

In [10]:
from langchain.chat_models import init_chat_model

model = init_chat_model("google_genai:gemini-2.5-flash-lite")
response = model.invoke("Why do parrots talk?")
response.content

'Parrots talk for a fascinating combination of **biological, social, and cognitive reasons**. It\'s not just a random phenomenon, but a complex behavior that serves them well in their natural environment and explains their remarkable ability to mimic human speech.\n\nHere\'s a breakdown of why parrots talk:\n\n**1. Social Bonding and Communication:**\n\n*   **Flock Animals:** Parrots are highly social creatures that live in flocks. In the wild, vocalizations are crucial for maintaining group cohesion, coordinating movements, warning of danger, and finding mates. Talking (or their natural vocalizations) is their primary way of interacting with their group.\n*   **Strengthening Bonds:** Just like humans use conversation to build relationships, parrots use vocalizations to strengthen their bonds with flock members. This can involve greetings, calls for reassurance, and sharing information.\n*   **Mimicry as a Social Tool:** In their natural environment, parrots often mimic the sounds arou

In [11]:
from langchain.tools import tool

@tool
def get_weather(location:str)-> str:
    """Get the weather at a given location"""
    return f"It is sunny in {location}"

model_with_tools = model.bind_tools([get_weather])

response = model_with_tools.invoke("What is the weather in Moscow?")
response.content

''

Now, here we can see we are not getting any content, so we will see how to parse this response in case of tool calls

In [12]:
print(response)

content='' additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "Moscow"}'}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019cf0b0-33cb-7d52-8362-176b6e4ffe3c-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Moscow'}, 'id': '8c66b076-86e7-4348-a9f3-a0bb5e72825f', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 47, 'output_tokens': 15, 'total_tokens': 62, 'input_token_details': {'cache_read': 0}}


Why this is happening?

By using .bind_tools([get_weather]), you told the model: "If you need weather info, use this tool."

When you asked about Moscow:

The model recognized it didn't know the live weather.

It generated a Tool Call request instead of a conversational sentence.

Since it is "calling" a function, it doesn't have a verbal response to give you yet—it's waiting for you to run the tool and give it the result.

In [13]:
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call["name"]}")
    print(f"Args: {tool_call['args']}")

Tool: get_weather
Args: {'location': 'Moscow'}


Here, this means , it didnt really call the tool yet, it generated the output which should be given to the tool thats it. When we execute model.invoke() again, we need to give the tool output, by running the tool ourselves and passing the output in this 2nd invoke call, then we will get the actual answer.

THIS IS NOTHING BUT SLOPPY DESIGN OF THE LIBRARY WHILE THEY BUILT IT.

#### Tool Execution loops

In [18]:
# STEP 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages) #This is just formatted response for calling the tool, thats why this does not have any content
messages.append(ai_msg)

Now, we can see the appended message, which could be used by the model.

Remember messages is a list

In [22]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "Boston"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019cf0c5-ecf0-7072-900f-fc836e2b6fb8-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': '9ff1691c-2806-4bf3-81a5-7fc28f35b8c6', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 48, 'output_tokens': 15, 'total_tokens': 63, 'input_token_details': {'cache_read': 0}})]

In [23]:
#STEP 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    #Execuet the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "Boston"}'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019cf0c5-ecf0-7072-900f-fc836e2b6fb8-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': '9ff1691c-2806-4bf3-81a5-7fc28f35b8c6', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 48, 'output_tokens': 15, 'total_tokens': 63, 'input_token_details': {'cache_read': 0}}),
 ToolMessage(content='It is sunny in Boston', name='get_weather', tool_call_id='9ff1691c-2806-4bf3-81a5-7fc28f35b8c6')]

Once we pass the messages list, which has the Aimessage and ToolMessage Objects as 2nd and 3rd elements, the model will recognize them and give us the results

In [24]:
#STEP 3: Pass results back to the model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

The weather in Boston is sunny.


#### This is what internal working of the tool calling which we have seen in the 1 notebook, where we were automatically getting the end results with tool calls nicely.